In [ ]:
## Gravitational Waves and Machine Learning
# Author: Nathan Steinle, Feb. 2026

# =============================================================
# This notebook is designed for a lecture on waves, GW detection theory, and
# a simple application of machine learning classification. 

# Takes about 3 minutes to run. 

# =============================================================
# Run this in a virtual python environment. Create one with the following commands (replace the path in MYENV):
#         MYVENV=<home path>/waves_env
#         virtualenv ${MYVENV}
#         source ${MYVENV}/bin/activate
#         pip3 install numpy scipy matplotlib pathos astropy pycbc scikit-learn jupyter jupyterlab




# =============================================================
# PART 0: Imports and Setup
# =============================================================

import numpy as np
import matplotlib.pyplot as plt

from pycbc.catalog import Merger
from pycbc.types import TimeSeries
from pycbc.filter import matched_filter
from pycbc.psd import interpolate, inverse_spectrum_truncation
from pycbc.waveform import get_td_waveform

np.random.seed(1234)




# =============================================================
# PART 1: Classical Matched Filtering (GWOSC LOSC tutorial)
# =============================================================

# -----------------------------
# 1. Load real LIGO data
# -----------------------------

m = Merger('GW150914')
strain = m.strain('H1')

# -----------------------------
# 2. Conditioning (LOSC tutorial)
# -----------------------------

strain = strain.crop(2, 2)
strain = strain.highpass_fir(30, 512)
strain = strain.resample(1.0 / 2048)

# -----------------------------
# 3. PSD estimation (LOSC tutorial)
# -----------------------------

psd = strain.psd(4)
psd = interpolate(psd, strain.delta_f)
psd = inverse_spectrum_truncation(
    psd,
    int(4 * strain.sample_rate),
    low_frequency_cutoff=30
)

# -----------------------------
# 4. Template waveform (LOSC tutorial)
# -----------------------------

template, _ = get_td_waveform(
    approximant='TaylorF2',
    mass1=36,
    mass2=29,
    delta_t=strain.delta_t,
    f_lower=30
)

# --- Match template length to data length (required for FFT)
if len(template) > len(strain):
    template = template.crop(0, template.duration - strain.duration)
elif len(template) < len(strain):
    pad = len(strain) - len(template)
    template = TimeSeries(
        np.pad(template.numpy(), (0, pad)),
        delta_t=template.delta_t,
        epoch=template.start_time
    )

assert len(template) == len(strain)

# -----------------------------
# 5. Matched filter (LOSC tutorial)
# -----------------------------

snr_ts = matched_filter(
    template,
    strain,
    psd=psd,
    low_frequency_cutoff=30
)

snr_ts = snr_ts.crop(4, 4)

snr = np.abs(snr_ts.numpy())
times = snr_ts.sample_times











In [ ]:
# =============================================================
# PART 2A: Generate a waveform bank
#          (late inspiral, merger, early ringdown)
# =============================================================

from pycbc.waveform import get_td_waveform
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

fs = 4096
delta_t = 1.0 / fs
f_lower = 30

n_bns = 200
n_bbh = 200
window_size = 4096   # ~1 second centered on merger

X = []
y = []

# -----------------------------
# Helper: extract merger-centered window
# -----------------------------
def extract_merger_window(h, window_size):
    peak = np.argmax(np.abs(h))
    half = window_size // 2

    start = max(0, peak - half)
    end = start + window_size

    if end > len(h):
        end = len(h)
        start = end - window_size

    h_win = h[start:end]

    if len(h_win) < window_size:
        h_win = np.pad(h_win, (0, window_size - len(h_win)))

    return h_win

# -----------------------------
# BNS waveforms (IMR + tidal effects)
# -----------------------------
for _ in range(n_bns):
    m1 = np.random.uniform(1.2, 1.6)
    m2 = np.random.uniform(1.2, 1.6)

    hp, _ = get_td_waveform(
        approximant="IMRPhenomD_NRTidal",
        mass1=m1,
        mass2=m2,
        delta_t=delta_t,
        f_lower=f_lower
    )

    h = extract_merger_window(hp.numpy(), window_size)

    X.append(h)
    y.append(0)   # BNS label

# -----------------------------
# BBH waveforms (full IMR)
# -----------------------------
for _ in range(n_bbh):
    m1 = np.random.uniform(8, 30)
    m2 = np.random.uniform(8, 30)

    hp, _ = get_td_waveform(
        approximant="IMRPhenomD",
        mass1=m1,
        mass2=m2,
        delta_t=delta_t,
        f_lower=f_lower
    )

    h = extract_merger_window(hp.numpy(), window_size)

    X.append(h)
    y.append(1)   # BBH label

X = np.array(X)
y = np.array(y)






print("Dataset shape:", X.shape)

# =============================================================
# PART 2B: Feature scaling and dimensionality reduction
#          Train / test split (RAW data first)
# =============================================================

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Log-amplitude scaling (dynamic range stabilization)
X_train_log = np.log10(np.abs(X_train_raw) + 1e-22)
X_test_log  = np.log10(np.abs(X_test_raw)  + 1e-22)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_log)
X_test_scaled  = scaler.transform(X_test_log)

# PCA captures dominant waveform morphology
pca = PCA(n_components=10)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print("Explained variance (first 5 PCs):",
      pca.explained_variance_ratio_[:5])




# =============================================================
# PART 2C: Supervised classification
# =============================================================

clf = LogisticRegression(max_iter=500)
clf.fit(X_train_pca, y_train)

y_pred = clf.predict(X_test_pca)

print(classification_report(
    y_test, y_pred,
    target_names=["BNS", "BBH"]
))




# =============================================================
# PART 2D: Visualization in PCA space (fixed for train/test split)
# =============================================================

# Combine train and test PCA data for a full view
X_full_pca = np.vstack([X_train_pca, X_test_pca])
y_full = np.hstack([y_train, y_test])

plt.figure(figsize=(7,6))
plt.scatter(
    X_full_pca[y_full==0, 0], X_full_pca[y_full==0, 1],
    label="BNS", alpha=0.7
)
plt.scatter(
    X_full_pca[y_full==1, 0], X_full_pca[y_full==1, 1],
    label="BBH", alpha=0.7
)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.title("BBH vs BNS Waveforms (Time Domain, PCA)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()






In [ ]:
# =============================================================
# Visualization: overlay true vs predicted labels (one column)
# =============================================================

n_plot = 10
idx = np.random.choice(len(X_test_raw), n_plot, replace=False)

time = np.arange(window_size) * delta_t

fig, axes = plt.subplots(n_plot, 1, figsize=(10, 2.0 * n_plot), sharex=True)

for i, k in enumerate(idx):
    label_true = "BBH" if y_test[k] == 1 else "BNS"
    label_pred = "BBH" if y_pred[k] == 1 else "BNS"

    # Plot true waveform
    axes[i].plot(time, X_test_raw[k], color='blue', lw=1.5, label=f"True: {label_true}")

    # Overlay predicted waveform (same strain, but line style shows label)
    axes[i].plot(time, X_test_raw[k],
                 color='red',
                 lw=1.2,
                 linestyle='--',
                 label=f"Predicted: {label_pred}")

    axes[i].set_ylabel("Strain")
    axes[i].legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel("Time (s)")

# plt.suptitle("BBH vs BNS Waveforms: True vs Predicted Labels (Overlay)", y=0.92)
plt.tight_layout()
plt.show()

In [ ]:

# =============================================================
# Cross-population generalization 
# =============================================================

MODE = "low_mass_train_high_mass_test"   # or "high_mass_train_low_mass_test"

def generate_dataset(bns_range, bbh_range, n_each=200):
    X_out, y_out = [], []

    # BNS
    for _ in range(n_each):
        m1 = np.random.uniform(*bns_range)
        m2 = np.random.uniform(*bns_range)

        hp, _ = get_td_waveform(
            approximant="TaylorF2",
            mass1=m1, mass2=m2,
            delta_t=delta_t,
            f_lower=f_lower
        )

        h = hp.numpy()[-window_size:]
        if len(h) < window_size:
            h = np.pad(h, (window_size - len(h), 0))

        X_out.append(h)
        y_out.append(0)

    # BBH
    for _ in range(n_each):
        m1 = np.random.uniform(*bbh_range)
        m2 = np.random.uniform(*bbh_range)

        hp, _ = get_td_waveform(
            approximant="IMRPhenomD",
            mass1=m1, mass2=m2,
            delta_t=delta_t,
            f_lower=f_lower
        )

        h = hp.numpy()[-window_size:]
        if len(h) < window_size:
            h = np.pad(h, (window_size - len(h), 0))

        X_out.append(h)
        y_out.append(1)

    return np.array(X_out), np.array(y_out)


if MODE == "low_mass_train_high_mass_test":
    X_train_raw, y_train = generate_dataset(
        bns_range=(1.2, 1.4),
        bbh_range=(8, 12)
    )
    X_test_raw, y_test = generate_dataset(
        bns_range=(1.4, 1.6),
        bbh_range=(20, 30)
    )

elif MODE == "high_mass_train_low_mass_test":
    X_train_raw, y_train = generate_dataset(
        bns_range=(1.4, 1.6),
        bbh_range=(20, 30)
    )
    X_test_raw, y_test = generate_dataset(
        bns_range=(1.2, 1.4),
        bbh_range=(8, 12)
    )

# --- Preprocessing (train → test, no leakage!)
X_train_log = np.log10(np.abs(X_train_raw) + 1e-22)
X_test_log  = np.log10(np.abs(X_test_raw)  + 1e-22)

X_train_scaled = scaler.fit_transform(X_train_log)
X_test_scaled  = scaler.transform(X_test_log)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

clf = LogisticRegression(max_iter=500)
clf.fit(X_train_pca, y_train)

y_pred = clf.predict(X_test_pca)

print(f"\nCross-population test: {MODE}")
print(classification_report(
    y_test, y_pred,
    target_names=["BNS", "BBH"]
))



